# 22 — Consolidated results: multi-concept concept-guided decoding

CPU-only analysis notebook: consolidates every completed experiment's JSONL into
report-ready tables, figures, and statistics. Reads (skipping any not yet run):

| experiment | file | design |
|---|---|---|
| E0 v2 — F1 vs F2 pilot | `resources/17_e0_pilot_v2.jsonl` | 3 strata × {F1.a, F2.a} × k∈{3,4} × 20 prompts |
| E2 — Riemannian means | `resources/18_e2_riemannian.jsonl` | 3 strata × {extrinsic, aligned, frechet} × 20 prompts |
| E2b — ρ crossover | `resources/19_e2b_crossover.jsonl` | 11 pairs × {extrinsic, aligned} × 20 prompts |
| E1 — family sweep | `resources/20_e1_family_sweep.jsonl` | 3 strata × 9 methods × 20 prompts |
| E3 — negative steering | `resources/21_e3_negative.jsonl` | 3 strata × 2 directions × 3 methods × 20 prompts |

All at k=4, steps=24 (E0 v2 also k=3), Meta-Llama-3.1-8B-Instruct-AWQ-INT4,
shared 20-prompt suite. Metric v2 throughout: Δexpression = post-hoc concept
projection of the continuation, steered − unsteered baseline. Outputs are saved
as `resources/22_*`.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats

SELECTION = json.loads(Path("resources/15_e0_selected_pairs.json").read_text())[
    "selected"
]
EXPECTED = {
    "17_e0_pilot_v2": 240,
    "18_e2_riemannian": 180,
    "19_e2b_crossover": 440,
    "20_e1_family_sweep": 540,
    "21_e3_negative": 360,
}


def load(name: str) -> list[dict] | None:
    path = Path(f"resources/{name}.jsonl")
    if not path.exists():
        print(f"MISSING  {name}: not run yet")
        return None
    records = [json.loads(line) for line in path.read_text().strip().split("\n")]
    status = "ok" if len(records) == EXPECTED[name] else "INCOMPLETE"
    print(f"{status:>10}  {name}: {len(records)}/{EXPECTED[name]} records")
    return records if len(records) == EXPECTED[name] else None


E0 = load("17_e0_pilot_v2")
E2 = load("18_e2_riemannian")
E2B = load("19_e2b_crossover")
E1 = load("20_e1_family_sweep")
E3 = load("21_e3_negative")

In [ ]:
def frame(records: list[dict], a_of: dict, b_of: dict, keys: tuple) -> pd.DataFrame:
    """Flatten harness records; a/b synsets resolved per record via callables."""
    rows = []
    for i, r in enumerate(records):
        c = r["config"]
        a, b = a_of(c), b_of(c)
        rows.append(
            {
                **{key: c.get(key) for key in keys},
                "prompt": r["prompt"],
                "a": a,
                "b": b,
                "delta_a": r["expression"][a]["delta"],
                "delta_b": r["expression"][b]["delta"],
                "ppl_ratio": r["ppl_ratio"],
                "fluency_flag": r["fluency_flag"],
                "text": r["text"],
                "row": i,
            }
        )
    return pd.DataFrame(rows)


def add_joint_z(df: pd.DataFrame) -> pd.DataFrame:
    """z per concept over the whole experiment; joint = min (AND semantics)."""
    for side in ("a", "b"):
        z = df.groupby(side)[f"delta_{side}"].transform(
            lambda g: (g - g.mean()) / (g.std() + 1e-8)
        )
        df[f"z_{side}"] = z
    df["joint_z"] = df[["z_a", "z_b"]].min(axis=1)
    return df


def by_stratum(side: str):
    return lambda c: SELECTION[c["stratum"]][side]


def by_pair(side: str):
    return lambda c: c["pair"].split(" / ")[0 if side == "a" else 1]

## 1. E0 v2 — frame-space vs score-space composition

The pre-registered hypothesis was that score-space F2.a *escapes* frame-averaging
interference at low ρ, so its advantage over F1.a should be largest there.
The paired per-prompt gap below tests that directly — and shows the opposite.

In [ ]:
if E0:
    e0 = add_joint_z(
        frame(E0, by_stratum("a"), by_stratum("b"), ("stratum", "rho", "method", "k"))
    )
    gaps = []
    for (stratum, k), group in e0.groupby(["stratum", "k"]):
        wide = group.pivot_table(
            index="prompt", columns="method", values="joint_z", aggfunc="first"
        )
        gap = wide["F2.a"] - wide["F1.a"]
        wilcoxon = stats.wilcoxon(wide["F2.a"], wide["F1.a"])
        gaps.append(
            {
                "stratum": stratum,
                "rho": group["rho"].iloc[0],
                "k": k,
                "gap_F2_minus_F1": gap.mean(),
                "sem": gap.sem(),
                "wilcoxon_p": wilcoxon.pvalue,
                "F1_wins": bool(gap.mean() < 0),
            }
        )
    e0_gaps = pd.DataFrame(gaps).sort_values(["k", "rho"])
    display(e0_gaps.round(4))
    print(
        "Reading: negative gap = F1.a (mean frame) beats F2.a (z-scored score sum).\n"
        "The gap does NOT become positive at low rho -> interference-escape via\n"
        "score space is contradicted; frame composition is the stronger family."
    )

## 2. E2 — does Riemannian aggregation matter? (RQ3 extrinsic test)

Three ways to average two concept frames into one steering frame:
**extrinsic** (chordal/Procrustes mean — the paper's), **frechet** (true Karcher
mean under the canonical Stiefel metric), **aligned** (generalized-Procrustes:
rotate frames within their spans first). Two questions: does the *metric*
matter (frechet vs extrinsic), and does the *gauge* matter (aligned vs both)?

In [ ]:
if E2:
    e2 = add_joint_z(
        frame(E2, by_stratum("a"), by_stratum("b"), ("stratum", "rho", "method"))
    )
    display(
        e2.groupby(["stratum", "rho", "method"])
        .agg(
            delta_a=("delta_a", "mean"),
            delta_b=("delta_b", "mean"),
            joint_z=("joint_z", "mean"),
            ppl_ratio=("ppl_ratio", "mean"),
        )
        .round(3)
        .sort_values("rho")
    )

    # metric vs gauge, quantified: identical-generation rate per method pair
    wide = e2.pivot_table(
        index=["stratum", "prompt"], columns="method", values="text", aggfunc="first"
    )
    for m1, m2 in (("extrinsic", "frechet"), ("extrinsic", "aligned")):
        same = (wide[m1] == wide[m2]).mean()
        print(f"byte-identical generations {m1} vs {m2}: {same:.0%}")
    print(
        "\nMetric barely matters (frechet ~ extrinsic); gauge does -- aligned"
        " rewrites most generations and wins the low-rho stratum."
    )

## 3. E2b — the ρ-adaptive crossover (headline figure)

11 concept pairs spanning ρ ∈ [0.19, 0.57], extrinsic vs aligned mean, paired
per prompt. Positive gap = aligned wins. The claim under test: **aligned wins
below ρ ≈ 0.30, extrinsic at or above it** — a practitioner-usable selection
rule computable *before* generating a single token.

In [ ]:
if E2B:
    e2b = frame(E2B, by_pair("a"), by_pair("b"), ("pair", "rho", "regime", "method"))
    e2b = add_joint_z(e2b)
    wide = e2b.pivot_table(
        index=["pair", "rho", "regime", "prompt"],
        columns="method",
        values="joint_z",
        aggfunc="first",
    ).reset_index()
    wide["gap"] = wide["aligned"] - wide["extrinsic"]

    per_pair = (
        wide.groupby(["pair", "rho", "regime"])["gap"]
        .agg(["mean", "sem", "count"])
        .reset_index()
        .sort_values("rho")
    )
    per_pair["wilcoxon_p"] = [
        stats.wilcoxon(g["aligned"], g["extrinsic"]).pvalue
        for _, g in wide.groupby(["pair", "rho", "regime"])
    ]
    display(per_pair.round(4))

    # sign consistency with the rho <= 0.30 rule
    rule = per_pair["rho"] < 0.30
    follows = ((per_pair["mean"] > 0) == rule).sum()
    n_pairs = len(per_pair)
    sign_p = stats.binomtest(follows, n_pairs, 0.5).pvalue
    print(
        f"pairs consistent with the rule: {follows}/{n_pairs}, sign test p = {sign_p:.4f}"
    )

    # inverse-variance weighted fit: polyfit multiplies w against the
    # UNSQUARED residual, so w = 1/sem realizes sum((resid/sem)^2)
    w = 1.0 / per_pair["sem"]
    slope, intercept = np.polyfit(per_pair["rho"], per_pair["mean"], 1, w=w)
    crossing = -intercept / slope
    print(
        f"weighted fit: gap = {slope:.2f}*rho + {intercept:.2f}; zero at rho = {crossing:.3f}"
    )

In [ ]:
if E2B:
    fig, ax = plt.subplots(figsize=(8.5, 5))
    regime_markers = {
        "unrelated": "o",
        "antonym": "D",
        "compositional": "s",
        "similar": "^",
    }
    for regime, group in per_pair.groupby("regime"):
        ax.errorbar(
            group["rho"],
            group["mean"],
            yerr=group["sem"],
            fmt=regime_markers[regime],
            ms=7,
            capsize=3,
            ls="",
            label=regime,
        )
    xs = np.linspace(per_pair["rho"].min() - 0.01, per_pair["rho"].max() + 0.01, 100)
    ax.plot(xs, slope * xs + intercept, "k--", lw=1, alpha=0.7)
    ax.axhline(0, color="gray", lw=0.8)
    ax.axvline(crossing, color="tab:red", lw=1, ls=":")
    ax.annotate(
        f"crossover\n rho = {crossing:.2f}",
        (crossing, ax.get_ylim()[1] * 0.75),
        color="tab:red",
        fontsize=9,
    )
    ax.set_xlabel("concept-pair correlation rho")
    ax.set_ylabel("paired joint-z gap  (aligned - extrinsic)")
    ax.set_title(
        f"Gauge alignment helps exactly where frames disagree "
        f"({follows}/{n_pairs} pairs sign-consistent, p={sign_p:.3f})"
    )
    ax.legend(title="pair regime", fontsize=8)
    plt.tight_layout()
    plt.savefig("resources/22_e2b_crossover_annotated.png", dpi=150)
    plt.show()

## 4. E1 — all families head-to-head (Pareto view)

Everything steers at 5–12× baseline perplexity, so single operating points
mislead: a method only "wins" if no other method matches its expression at
lower fluency cost. Heatmap of joint-z per method × stratum, plus each
stratum's Pareto set over (joint expression, PPL ratio).

In [ ]:
if E1:
    e1 = add_joint_z(
        frame(
            E1,
            by_stratum("a"),
            by_stratum("b"),
            ("stratum", "rho", "method", "family"),
        )
    )
    e1_summary = (
        e1.groupby(["stratum", "rho", "method", "family"])
        .agg(
            joint_z=("joint_z", "mean"),
            delta_a=("delta_a", "mean"),
            delta_b=("delta_b", "mean"),
            ppl_ratio=("ppl_ratio", "mean"),
        )
        .reset_index()
    )

    heat = e1_summary.pivot_table(
        index="method", columns="stratum", values="joint_z"
    ).reindex(columns=["low", "medium", "high"])
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    im = ax.imshow(heat.values, cmap="RdYlGn", aspect="auto")
    ax.set_xticks(range(3), heat.columns)
    ax.set_yticks(range(len(heat)), heat.index)
    for i in range(len(heat)):
        for j in range(3):
            ax.text(
                j, i, f"{heat.values[i, j]:.2f}", ha="center", va="center", fontsize=8
            )
    ax.set_title("E1: mean joint-z by method and rho stratum")
    fig.colorbar(im, shrink=0.8)
    plt.tight_layout()
    plt.savefig("resources/22_e1_method_heatmap.png", dpi=150)
    plt.show()

    def pareto(points):
        keep = []
        for _, row in points.iterrows():
            dominated = (
                (points["ppl_ratio"] <= row["ppl_ratio"])
                & (points["joint_z"] >= row["joint_z"])
                & (
                    (points["ppl_ratio"] < row["ppl_ratio"])
                    | (points["joint_z"] > row["joint_z"])
                )
            ).any()
            keep.append(not dominated)
        return points[keep].sort_values("ppl_ratio")

    print("Pareto set (joint-z vs PPL ratio) per stratum:")
    for stratum in ("low", "medium", "high"):
        members = pareto(e1_summary[e1_summary["stratum"] == stratum])
        listing = ", ".join(
            f"{r['method']} ({r['joint_z']:.2f} @ {r['ppl_ratio']:.1f}x)"
            for _, r in members.iterrows()
        )
        print(f"  {stratum:>6}: {listing}")

## 5. E3 — negative / mixed steering

Attract one concept, repel the other. Two-sided success: Δ(attract) > 0 AND
Δ(repel) < 0. `attract-only` is the incidental-suppression control the negative
methods must beat.

In [ ]:
if E3:
    e3 = frame(
        E3,
        lambda c: c["attract"],
        lambda c: c["repel"],
        ("stratum", "rho", "method", "attract", "repel"),
    ).rename(columns={"delta_a": "delta_attract", "delta_b": "delta_repel"})
    e3["clean"] = (e3["delta_attract"] > 0) & (e3["delta_repel"] < 0)
    e3_summary = (
        e3.groupby(["stratum", "rho", "method"])
        .agg(
            delta_attract=("delta_attract", "mean"),
            delta_repel=("delta_repel", "mean"),
            clean_rate=("clean", "mean"),
            ppl_ratio=("ppl_ratio", "mean"),
        )
        .reset_index()
        .sort_values(["rho", "method"])
    )
    display(e3_summary.round(3))

    # paired suppression vs control, pooled over strata and directions
    wide = e3.pivot_table(
        index=["stratum", "attract", "prompt"],
        columns="method",
        values="delta_repel",
        aggfunc="first",
    ).dropna()
    for method in ("frame-diff", "score-neg"):
        gap = wide[method] - wide["attract-only"]
        p = stats.wilcoxon(wide[method], wide["attract-only"]).pvalue
        print(
            f"{method} suppression beyond control: {gap.mean():+.3f}"
            f" ({(gap < 0).mean():.0%} of prompts, Wilcoxon p = {p:.2g})"
        )

## 6. Fluency panorama

The cost side of every result above: continuation-PPL ratio vs the model's own
greedy baseline (structurally harsh — the baseline is the PPL floor — so these
support *relative* comparisons, not absolute quality claims). The 2.5×
guardrail is far below every method at k=4.

In [ ]:
panels = {
    "E0v2 (k=4)": e0[e0["k"] == 4].assign(label=lambda d: d["method"]) if E0 else None,
    "E2": e2.assign(label=lambda d: d["method"]) if E2 else None,
    "E1": e1.assign(label=lambda d: d["method"]) if E1 else None,
    "E3": e3.assign(label=lambda d: d["method"]) if E3 else None,
}
panels = {name: df for name, df in panels.items() if df is not None}

fig, axes = plt.subplots(1, len(panels), figsize=(4.2 * len(panels), 4.4), sharey=True)
axes = np.atleast_1d(axes)
for ax, (name, df) in zip(axes, panels.items()):
    order = df.groupby("label")["ppl_ratio"].median().sort_values().index
    ax.boxplot(
        [df[df["label"] == label]["ppl_ratio"] for label in order],
        tick_labels=order,
        showfliers=False,
    )
    ax.axhline(2.5, color="tab:red", ls="--", lw=0.8)
    ax.set_title(name)
    ax.tick_params(axis="x", rotation=90, labelsize=7)
axes[0].set_ylabel("PPL ratio (steered / unsteered greedy)")
plt.tight_layout()
plt.savefig("resources/22_fluency_overview.png", dpi=150)
plt.show()

## 7. Master summary table

One row per (experiment, stratum/pair, method): the numbers behind every claim,
exported for the report.

In [ ]:
blocks = []
if E0:
    blocks.append(
        e0.groupby(["stratum", "rho", "method", "k"])
        .agg(joint_z=("joint_z", "mean"), ppl_ratio=("ppl_ratio", "mean"))
        .reset_index()
        .assign(experiment="E0v2")
    )
if E2:
    blocks.append(
        e2.groupby(["stratum", "rho", "method"])
        .agg(joint_z=("joint_z", "mean"), ppl_ratio=("ppl_ratio", "mean"))
        .reset_index()
        .assign(experiment="E2")
    )
if E2B:
    blocks.append(
        e2b.groupby(["pair", "rho", "regime", "method"])
        .agg(joint_z=("joint_z", "mean"), ppl_ratio=("ppl_ratio", "mean"))
        .reset_index()
        .assign(experiment="E2b")
    )
if E1:
    blocks.append(e1_summary.assign(experiment="E1"))
if E3:
    blocks.append(e3_summary.assign(experiment="E3"))

master = pd.concat(blocks, ignore_index=True)
master.to_csv("resources/22_master_summary.csv", index=False)
print(f"master summary: {len(master)} rows -> resources/22_master_summary.csv")
master.round(3).head(15)

## 8. Qualitative showcase

Same prompt ("Tell me a short story."), low-ρ pair (sadness + music), across
composition methods — the examples that make the numbers legible in a talk.

In [ ]:
def answer(text: str) -> str:
    return text.split("assistant<|end_header_id|>")[-1].strip()


def showcase(records, method_key, methods, label):
    for r in records:
        c = r["config"]
        if c.get("stratum", "") == "low" and "short story" in r["prompt"]:
            if c[method_key] in methods:
                deltas = {
                    k.split(".")[0]: round(v["delta"], 2)
                    for k, v in r["expression"].items()
                }
                print(f"[{label} / {c[method_key]}] {deltas}")
                print("   ", answer(r["text"])[:160], "\n")


if E2:
    showcase(E2, "method", {"extrinsic", "aligned"}, "E2")
if E1:
    showcase(E1, "method", {"F2.b-softmin", "F3.c-sentence", "beam-F1.a"}, "E1")
if E3:
    showcase(E3, "method", {"frame-diff", "score-neg"}, "E3")